## 1. Install and Import

In [1]:
# Run this once
!pip install selenium webdriver-manager

  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached typing_extensions-4.14.1-py3-none-any.whl.metadata (3.0 kB)
Using cached typing_extensions-4.14.1-py3-none-any.whl (43 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)

  Attempting uninstall: urllib3

    Found existing installation: urllib3 1.26.20

    Uninstalling urllib3-1.26.20:

      Successfully uninstalled urllib3-1.26.20

   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
   ---------------------------------------- 0/2 [urllib3]
  Attempting uninstall: typing_extensions
   ---------------------------------------- 0/2 [urllib3]
    Found existing installation: typing_extensions 4.15.0
   ---------------------------------------- 0/2 [urllib3]
    Uninstalling typing_extensions-4.15.0:
   ---------------------------------------- 0/2 [urllib3]
   -------------------- -----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.3.68 requires packaging<25,>=23.2, but you have packaging 25.0 which is incompatible.
langchain-openai 0.3.28 requires openai<2.0.0,>=1.86.0, but you have openai 1.55.3 which is incompatible.
pyppeteer 2.0.0 requires pyee<12.0.0,>=11.0.0, but you have pyee 13.0.0 which is incompatible.
pyppeteer 2.0.0 requires urllib3<2.0.0,>=1.25.8, but you have urllib3 2.6.3 which is incompatible.
selenium-wire 5.1.0 requires h2>=4.0; python_version >= "3.6.0", but you have h2 3.2.0 which is incompatible.
selenium-wire 5.1.0 requires hyperframe>=6.0; python_version >= "3.6.0", but you have hyperframe 5.2.0 which is incompatible.
seleniumbase 4.41.3 requires chardet==5.2.0, but you have chardet 3.0.4 which is incompatible.
seleniumbase 4.41.3 requires idna==3.10, but you have idna 2.10 which is incompatible.
sele

In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

## 2. Launch Browser
We use `headless=True` so no browser window opens. Remove that option if you want to watch it live.

In [6]:
def get_driver(headless=True):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    return driver

driver = get_driver(headless=True)
print("Browser launched.")

Browser launched.


## 3. Login Automation
Using `quotes.toscrape.com/login` — a login page built for practice.

Credentials: `username=admin`, `password=12345`

In [7]:
driver.get("http://quotes.toscrape.com/login")
time.sleep(1)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: Quotes to Scrape
Current URL: https://quotes.toscrape.com/login


In [8]:
# Find fields and fill them
username_field = driver.find_element(By.ID, "username")
password_field = driver.find_element(By.ID, "password")

username_field.clear()
username_field.send_keys("admin")

password_field.clear()
password_field.send_keys("12345")

# Click login button
login_btn = driver.find_element(By.CSS_SELECTOR, "input[type='submit']")
login_btn.click()

time.sleep(1)
print("Logged in.")
print("Current URL:", driver.current_url)

Logged in.
Current URL: https://quotes.toscrape.com/


In [9]:
# Verify login — check if logout link appears
try:
    logout = driver.find_element(By.XPATH, "//a[contains(text(),'Logout')]")
    print("Login successful — Logout button found.")
except:
    print("Login may have failed.")

Login successful — Logout button found.


## 4. Data Extraction — Quotes and Authors
After login, scrape quotes from multiple pages.

In [11]:
def scrape_page(driver):
    quotes = driver.find_elements(By.CLASS_NAME, "quote")
    data = []
    for q in quotes:
        text   = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        tags   = [t.text for t in q.find_elements(By.CLASS_NAME, "tag")]
        data.append({"quote": text, "author": author, "tags": ", ".join(tags)})
    return data

all_data = []
page = 1

while page <= 3:
    driver.get(f"http://quotes.toscrape.com/page/{page}/")
    time.sleep(1)
    page_data = scrape_page(driver)
    all_data.extend(page_data)
    print(f"Page {page}: {len(page_data)} quotes scraped")
    page += 1

print(f"Total quotes collected: {len(all_data)}")

Page 1: 10 quotes scraped
Page 2: 10 quotes scraped
Page 3: 10 quotes scraped
Total quotes collected: 30


In [12]:
df = pd.DataFrame(all_data)
print(df.shape)
df.head()

(30, 3)


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"


## 5. Data Extraction — Book Titles and Prices
Scraping product data from `books.toscrape.com`.

In [14]:
driver.get("http://books.toscrape.com")
time.sleep(1)

books = []
for page in range(1, 4):
    driver.get(f"http://books.toscrape.com/catalogue/page-{page}.html")
    time.sleep(1)

    items = driver.find_elements(By.CSS_SELECTOR, "article.product_pod")
    for item in items:
        title  = item.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
        price  = item.find_element(By.CLASS_NAME, "price_color").text
        rating = item.find_element(By.CSS_SELECTOR, "p.star-rating").get_attribute("class").split()[-1]
        books.append({"title": title, "price": price, "rating": rating})

    print(f"Page {page}: {len(items)} books")

books_df = pd.DataFrame(books)
print(f"Total books: {len(books_df)}")
books_df.head()

Page 1: 20 books
Page 2: 20 books
Page 3: 20 books
Total books: 60


,title,price,rating
0,A Light in the Attic,£51.77,Three
1,Tipping the Velvet,£53.74,One
2,Soumission,£50.10,One
3,Sharp Objects,£47.82,Four
4,Sapiens: A Brief History of Humankind,£54.23,Five


## 6. Save and Analyze Extracted Data

In [ ]:
# Save to CSV
df.to_csv("quotes.csv", index=False)
books_df.to_csv("books.csv", index=False)
print("Data saved to quotes.csv and books.csv")

# Quick analysis
print("
Top 5 authors by quote count:")
print(df['author'].value_counts().head())

print("
Books by rating:")
print(books_df['rating'].value_counts())

Data saved to quotes.csv and books.csv

Top 5 authors by quote count:
Albert Einstein      3
J.K. Rowling         2
Mark Twain           2
Marilyn Monroe       1
André Gide           1

Books by rating:
Three     14
One       12
Four      11
Five      11
Two       12

## 7. Taking a Screenshot

In [ ]:
driver.get("http://books.toscrape.com")
time.sleep(1)
driver.save_screenshot("screenshot.png")
print("Screenshot saved as screenshot.png")

Screenshot saved as screenshot.png

## 8. Close the Browser

In [ ]:
driver.quit()
print("Browser closed.")

Browser closed.

## Summary

| Task | Method Used |
|---|---|
| Launch browser | `webdriver.Chrome()` |
| Open URL | `driver.get(url)` |
| Find element | `find_element(By.ID / By.CLASS_NAME / By.CSS_SELECTOR)` |
| Type text | `.send_keys()` |
| Click | `.click()` |
| Extract text | `.text` |
| Extract attribute | `.get_attribute('href')` |
| Screenshot | `driver.save_screenshot()` |
| Close | `driver.quit()` |

**Note:** Always call `driver.quit()` at the end to free memory.